# DSA 405 P2 — House PTR PDF extraction and cleaning audit

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/StrokeOfLuck/dsa405-part-2/blob/main/notebooks/DSA405_002_FA26_P2_sryan3.ipynb)

**What changes from P1:** Start with the archived **2025 House filing PDFs**. The original scraper's pinned Stage 3 turns those PDFs into a transaction CSV; Stage 4 resolves ticker and asset fields into a second CSV. This P2 notebook audits Stage 3 and logs the Stage 4 decisions. The published scraper repo is never modified.

**Unit:** one transaction row extracted from a PDF. `source_year=2025` is the filing index year, not necessarily the transaction year.

**Scope confirmed:** filings indexed under 2025, including any earlier transaction dates.
The raw layout is a PDF report with repeated page headers, multi-line transaction descriptions,
and filing-level information outside the transaction table. It is not one rectangular table
with one observation per row. The extraction produces one reported transaction per row;
filing ID and transaction number identify an extracted row, not a unique economic event.

**Review boundary:** Sean checked four selected PDF cases (five transaction rows) with
ChatGPT assistance. This is targeted review, not a representative sample or verification of
every row. Exactly one approved amount correction is applied below, with old values retained.
All other unresolved cases remain flagged. No committee join is part of P2.


## Setup: Make an isolated 2025 copy and rebuild the CSVs

The pinned source commit and checksums in `data/raw/2025_pdf_manifest.csv` identify 515 PDFs. The complete raw snapshot is committed unmodified in `data/raw/2025_pdfs/`, with the 2025 House XML index at `data/raw/2025FD.xml`. The script verifies those files against the manifest, then copies them into `data/work/` before parsing. Git/network access is still needed on the first run to fetch the pinned parser source code. It may take several minutes on CPU.

**What to look for:** `Pipeline result: 0` means the script finished. The run verifies the committed raw PDFs, copies them into `data/work/`, and parses the working copies. If it fails, open `data/work/rebuild.log` for the full error. Nothing in this cell writes to `data/raw/` or to the original scraper repository.

In [1]:
# Bring in tools we need:
# Path works with file and folder names; os moves between folders.
# subprocess runs a command (like Git) from Python; sys tells us which Python is running.
from pathlib import Path
import os, subprocess, sys

# Find the rest of this project. Colab opens the notebook, but may not have its files yet.
if not Path("scripts/rebuild_2025.py").exists():
    # If we started inside the notebooks folder, go up one folder.
    if Path("../scripts/rebuild_2025.py").exists():
        os.chdir("..")
    else:
        # Otherwise, download a copy of this GitHub project into dsa405-part-2.
        target = Path("dsa405-part-2")
        if not (target / "scripts/rebuild_2025.py").exists():
            # Run Git only if the copy is missing. check=True stops if the download fails.
            subprocess.run(["git","clone","--depth","1","https://github.com/StrokeOfLuck/dsa405-part-2.git",str(target)],check=True)
        os.chdir(target)

# Try importing a Colab-only tool to see whether this notebook is running in Colab.
try:
    import google.colab  # available in Colab, absent in a normal local notebook
except ImportError:
    # Outside Colab, there is nothing to install automatically here.
    pass
else:
    # In Colab, install the packages listed in requirements.txt.
    # sys.executable makes sure we install them for the Python running this notebook.
    subprocess.run([sys.executable,"-m","pip","install","-q","-r","requirements.txt"],check=True)

import hashlib
def file_hash(p):
    return hashlib.sha256(p.read_bytes()).hexdigest()
raw_hashes_before = {str(p): file_hash(p) for p in Path("data/raw").rglob("*") if p.is_file()}

# Save the rebuild's messages in a text file called rebuild.log.
log_path=Path("data/work/rebuild.log")
log_path.parent.mkdir(parents=True,exist_ok=True)
# Run the rebuild script: it checks the PDFs and makes the Stage 3 and Stage 4 CSVs.
# Send its messages and errors to the log instead of printing pages of output here.
with log_path.open("w") as log:
    result=subprocess.run([sys.executable,"scripts/rebuild_2025.py"],stdout=log,stderr=subprocess.STDOUT)
# A result of 0 means the script finished successfully. Show its last 18 messages.
print("Pipeline result:",result.returncode)
print("\n".join(log_path.read_text(errors="replace").splitlines()[-18:]))
# If the script failed, stop here so later cells do not read old or incomplete CSVs.
if result.returncode:
    raise RuntimeError(f"Pipeline failed. Inspect {log_path} for the stage and error.")

Pipeline result: 0

Published website CSV:
C:\Users\Sean\Documents\Codex\2026-09-25\so-x20\work\finish-p2\data\work\06_public\house_ptr_transactions_web.csv
7,667 rows x 20 columns

Published metadata:
C:\Users\Sean\Documents\Codex\2026-09-25\so-x20\work\finish-p2\data\work\06_public\house_ptr_metadata.json
{
  "updated_at": "2026-09-25T23:43:35-04:00",
  "transactions": 7667,
  "public_table_rows": 7667,
  "public_table_columns": 20,
  "source_pdfs_accounted_for": 515,
  "fallback_pdfs": 66,
  "full_csv_filename": "house_ptr_transactions_latest.csv",
  "web_csv_filename": "house_ptr_transactions_web.csv"
}
P2 outputs: C:\Users\Sean\Documents\Codex\2026-09-25\so-x20\work\finish-p2\data\work\04_transactions\transactions_raw.csv C:\Users\Sean\Documents\Codex\2026-09-25\so-x20\work\finish-p2\data\work\04_transactions\transactions_resolved.csv


The raw PDFs committed in `data/raw/` remain untouched; working copies are checked against the committed manifest before parsing. Stage 3 writes `transactions_raw.csv`; Stage 4 writes `transactions_resolved.csv`. Publication makes a web-facing CSV too. No source PDF is changed.

## 1. Systematic audit of extracted source data

**What to look for:** the expected rebuild has 515 copied PDFs and 7,667 transaction rows in each CSV. Counts describe this pinned snapshot; investigate if a future input changes them.

In [2]:
# pandas reads CSV files into tables called DataFrames.
# display shows a table neatly inside the notebook.
import pandas as pd
from IPython.display import display

# Name the two CSV files created by the previous cell.
# Stage 3 is the first extraction; Stage 4 is the later ticker check.
source=Path("data/work/04_transactions/transactions_raw.csv")
resolved=Path("data/work/04_transactions/transactions_resolved.csv")
# Load both tables. dtype=str keeps IDs and printed text exactly as written.
# keep_default_na=False keeps a blank cell as "" instead of turning it into a special missing value.
raw=pd.read_csv(source,dtype=str,keep_default_na=False)
clean=pd.read_csv(resolved,dtype=str,keep_default_na=False)
# .shape gives (number of rows, number of columns) for each table.
print("V8.1 extracted:",raw.shape,"V8.2 resolved:",clean.shape)
# Count the PDF files copied into our working folder. We expect 515 in this snapshot.
print("Source PDFs copied:",len(list(Path("data/work/01_pdfs/2025").glob("*.pdf"))))
# Count filing-index years. A filing indexed in 2025 can report an earlier trade.
print("Source year:",raw.source_year.value_counts(dropna=False).to_dict())
# Show the first five rows and a few useful columns so we can see what one trade looks like.
display(raw[["filing_id","transaction_number_in_filing","politician","asset","ticker","transaction_date","amount_category","needs_review"]].head(5))
# Keep every dictionary/audit row and complete category lists visible in saved outputs.
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
assert len(raw) == len(clean)
assert raw[["filing_id", "transaction_number_in_filing"]].equals(clean[["filing_id", "transaction_number_in_filing"]])


V8.1 extracted: (7667, 50) V8.2 resolved: (7667, 61)
Source PDFs copied: 515
Source year: {'2025': 7667}


,filing_id,transaction_number_in_filing,politician,asset,ticker,transaction_date,amount_category,needs_review
0,20016861,1,Hon. John McGuire,UnitedHealth Group Incorporated Common Stock,UNH,2025-04-10,"$1,001 - $15,000",False
1,20018054,1,Hon. James Comer,Alphabet Inc. - Class C Capital Stock,GOOG,2025-01-02,"$1,001 - $15,000",False
2,20018054,2,Hon. James Comer,"Amazon.com, Inc. - Common Stock",AMZN,2025-01-02,"$1,001 - $15,000",False
3,20018054,3,Hon. James Comer,"Amazon.com, Inc. - Common Stock",AMZN,2025-01-02,"$1,001 - $15,000",False
4,20018054,4,Hon. James Comer,Amphenol Corporation Common Stock,APH,2025-01-02,"$1,001 - $15,000",False


### Column inventory

Read identifiers as strings to preserve their printed form. Empty strings count as missing. Audit every field used in this notebook before looking at the resolved output.

**What to look for:** a missing value here is exactly `""` in the CSV; it does not by itself mean the underlying PDF lacked the information.

In [3]:
# Audit the extraction plus every field introduced by the resolver.
# Strings preserve IDs, blanks, and original text; conversions here are diagnostic only.
extra = [x for x in clean.columns if x not in raw.columns]
audit_input = pd.concat([raw, clean[extra]], axis=1)
FIELDS = list(audit_input.columns)
BOOL_FIELDS = set("page_recovery_used filing_page_completeness_issue page_continuation_used possible_adjacent_same_signature row_shaded unicode_repair_applied needs_review needs_review_v8_1 ticker_changed asset_changed_by_ticker_resolver raw_date_has_extra_text date_prefix_disagrees".split())
INT_FIELDS = set("source_year transaction_number_in_filing page last_page duplicate_geometry_rows_merged unicode_repair_count".split())
MONEY_FIELDS = set("amount_min amount_max amount_exact amount_exact_before_p2".split())
SCORE_FIELDS = {"geometry_quality_score", "geometry_quality_score_v8_1"}
DATE_FIELDS = {"transaction_date", "notification_date"}
CATEGORY_FIELDS = set("member_status state_district owner asset_type transaction_type amount_category amount_status filing_status page_parse_method review_level review_level_v8_1 ticker_parse_status ticker_validation_source amount_status_before_p2".split())
def field_type(x):
    if x in BOOL_FIELDS: return "boolean"
    if x in INT_FIELDS: return "integer"
    if x in MONEY_FIELDS or x in SCORE_FIELDS: return "float"
    if x in DATE_FIELDS: return "date"
    if x in CATEGORY_FIELDS: return "category"
    return "string"
TYPES = {x: field_type(x) for x in FIELDS}
audit = pd.DataFrame([{
    "variable": x, "loaded dtype": str(audit_input[x].dtype), "intended type": TYPES[x],
    "missing count": int(audit_input[x].eq("").sum()),
    "missing rate": round(audit_input[x].eq("").mean(), 4),
    "distinct including blank": int(audit_input[x].nunique(dropna=False))
} for x in FIELDS])
display(audit)


,variable,loaded dtype,intended type,missing count,missing rate,distinct including blank
0,filing_id,object,string,0,0.0000,449
1,politician,object,string,0,0.0000,97
2,member_status,object,category,0,0.0000,2
3,state_district,object,category,0,0.0000,96
4,source_pdf,object,string,0,0.0000,449
5,source_year,object,integer,0,0.0000,1
6,transaction_number_in_filing,object,integer,0,0.0000,722
7,owner,object,category,3799,0.4955,4
8,asset,object,string,0,0.0000,1656
9,ticker,object,string,744,0.0970,1148


**What to look for:** `transaction_date_raw` can fail strict standalone-date parsing because neighboring PDF text was captured; inspect examples in the next cell before calling the normalized date incorrect.

In [4]:
# Show every observed value for short categorical domains, including blanks.
levels = []
for field in FIELDS:
    if field_type(field) in {"category", "boolean"} and audit_input[field].nunique(dropna=False) < 30:
        levels.append({"variable": field, "levels and counts": audit_input[field].value_counts(dropna=False).to_dict()})
display(pd.DataFrame(levels))

numeric_checks = []
for field in sorted((INT_FIELDS | MONEY_FIELDS | SCORE_FIELDS) & set(FIELDS)):
    s = audit_input[field]
    parsed = pd.to_numeric(s.replace("", pd.NA), errors="coerce")
    numeric_checks.append({"variable": field, "min": parsed.min(), "max": parsed.max(),
        "range": parsed.max()-parsed.min(), "nonblank invalid": int((parsed.isna() & s.ne("")).sum())})
display(pd.DataFrame(numeric_checks))

date_checks = []
for field, fmt in [("transaction_date", "%Y-%m-%d"), ("notification_date", "%Y-%m-%d"),
                   ("transaction_date_raw", "%m/%d/%Y"), ("notification_date_raw", "%m/%d/%Y")]:
    s = raw[field]
    parsed = pd.to_datetime(s.replace("", pd.NA), format=fmt, errors="coerce")
    date_checks.append({"variable": field, "min": parsed.min(), "max": parsed.max(),
                        "nonblank unparsed": int((parsed.isna() & s.ne("")).sum())})
display(pd.DataFrame(date_checks))


,variable,levels and counts
0,member_status,"{'Member': 7665, 'Former Member': 2}"
1,owner,"{'': 3799, 'SP': 2466, 'JT': 1191, 'DC': 211}"
2,asset_type,"{'ST': 6830, 'GS': 460, 'OT': 172, 'CS': 60, 'HN': 41, 'CT': 32, 'OP': 27, 'OI': 21, 'PS': 10, 'ET': 5, 'AB': 4, 'OL': 2, 'VA': 2, 'RS': 1}"
3,transaction_type,"{'P': 3836, 'S': 2760, 'S (partial)': 1048, 'E': 23}"
4,amount_category,"{'$1,001 - $15,000': 5822, '$15,001 - $50,000': 1118, '$50,001 - $100,000': 319, '$100,001 - $250,000': 214, '$250,001 - $500,000': 102, '$500,001 - $1,000,000': 42, '$1,000,001 - $5,000,000': 31, '': 10, '$5,000,001 - $25,000,000': 8, '$25,000,001 - $50,000,000': 1}"
5,amount_status,"{'valid_range': 7657, 'nonstandard_exact': 8, 'missing_range_bound': 2}"
6,filing_status,"{'New': 7582, 'New $200?': 75, 'Amended': 5, 'Deleted': 4, '': 1}"
7,page_parse_method,"{'table': 7480, 'relaxed_table': 182, 'date_anchor_recovery': 5}"
8,page_recovery_used,"{'False': 7456, 'True': 211}"
9,filing_page_completeness_issue,{'False': 7667}


,variable,min,max,range,nonblank invalid
0,amount_exact,224.0,2761.57,2537.57,0
1,amount_max,15000.0,50000000.00,49985000.00,0
2,amount_min,1001.0,25000001.00,24999000.00,0
3,duplicate_geometry_rows_merged,0.0,0.00,0.00,0
4,geometry_quality_score,43.0,100.00,57.00,0
5,geometry_quality_score_v8_1,43.0,100.00,57.00,0
6,last_page,1.0,81.00,80.00,0
7,page,1.0,81.00,80.00,0
8,source_year,2025.0,2025.00,0.00,0
9,transaction_number_in_filing,1.0,722.00,721.00,0


,variable,min,max,nonblank unparsed
0,transaction_date,2015-05-08,2025-12-30,0
1,notification_date,1935-03-28,2025-12-30,0
2,transaction_date_raw,2015-05-08,2025-12-29,640
3,notification_date_raw,1935-03-28,2025-12-30,235


**What to look for:** 640 raw-date strings fail strict standalone parsing in this snapshot; 617 were unflagged. This check identifies an extraction-text issue; it does not prove that the normalized transaction date is wrong.

In [5]:
# One filing can list several trades. Together, filing ID and row number should identify one trade.
keys=["filing_id","transaction_number_in_filing"]
# Collect the results in a named list of checks so the output is easy to read.
checks={
 # Count rows whose every field matches another row; these could inflate totals.
 "exact duplicates":int(raw.duplicated().sum()),
 # Count repeated filing ID + row number combinations, even if other fields differ.
 "duplicate filing+row keys":int(raw.duplicated(keys).sum()),
 # Look for placeholder words like n/a and 999 that may stand in for missing data.
 "sentinel tokens":int(sum(audit_input[x].str.strip().str.lower().isin(["n/a","na","null","none","999"]).sum() for x in FIELDS)),
 # Look for �, a character that can appear when PDF text is decoded badly.
 "replacement character (encoding)":int(sum(audit_input[x].str.contains("\ufffd",regex=False).sum() for x in FIELDS)),
 # Check IDs starting with 0; reading IDs as text preserves that leading zero.
 "leading-zero IDs":int(raw.filing_id.str.match(r"^0[0-9]+$").sum()),
 # A PDF table might have a total line that should not be counted as a trade.
 "total/subtotal labels":int(sum(audit_input[x].str.contains(r"^\s*(?:total|subtotal)\s*$",case=False,regex=True).sum() for x in ["asset","politician"])),
 # The bottom of a dollar range should never be larger than its top.
 "nonempty amount min > max":int((pd.to_numeric(raw.amount_min,errors="coerce")>pd.to_numeric(raw.amount_max,errors="coerce")).sum()),
 # Count rows already marked for review by the original parser.
 "review flagged":int(raw.needs_review.eq("True").sum()),
}
# Turn the named counts into a small table.
display(pd.Series(checks,name="count").to_frame())
# Try reading the entire original date field as just MM/DD/YYYY.
# Extra text from a neighboring PDF cell will cause this strict check to fail.
raw_date=pd.to_datetime(raw.transaction_date_raw.replace("",pd.NA),format="%m/%d/%Y",errors="coerce")
# Keep only nonblank raw dates that failed the strict check.
bad_date=raw.transaction_date_raw.ne("")&raw_date.isna()
# Count those dates and how many the original parser did not flag.
print("Non-standalone raw date strings:",int(bad_date.sum()),"of which scraper did not flag:",int((bad_date&raw.needs_review.eq("False")).sum()))
# Show eight examples with PDF links. A failed raw-text check does not prove
# the cleaned transaction date is wrong; inspect the source PDF.
display(raw.loc[bad_date,["filing_id","transaction_number_in_filing","transaction_date_raw","transaction_date","needs_review","original_pdf_url"]].head(8))
not_found = [name for name, count in checks.items() if count == 0]
print("Checked and not found under these definitions:", "; ".join(not_found))
print("These token/pattern checks are limited screens, not proof that every form of encoding or missing-value defect is absent.")
print("Transactions before 2025 retained:", int(raw.transaction_date.str[:4].lt("2025").sum()))
print("Blank, exact, and ranged amounts are different states:")
display(raw.amount_status.value_counts(dropna=False).rename_axis("amount_status").to_frame("rows"))
print("Amount exceptions requiring source review:")
display(raw.loc[raw.amount_status.ne("valid_range"), ["filing_id", "transaction_number_in_filing", "amount_raw", "amount_exact", "amount_status", "original_pdf_url"]])
print("Adjacent similar rows flagged (not automatically duplicates):", int(raw.possible_adjacent_same_signature.eq("True").sum()))

# Quantify contradictions to the domains we can actually test.
# Source descriptions and the economic meaning of codes still require PDF/codebook review.
def domain_failures(frame, field):
    s = frame[field].astype(str)
    present = s.ne("")
    if field in BOOL_FIELDS:
        return present & ~s.isin(["True", "False"])
    if field in INT_FIELDS | MONEY_FIELDS | SCORE_FIELDS:
        value = pd.to_numeric(s.replace("", pd.NA), errors="coerce")
        bad = present & (value.isna() | value.lt(0))
        if field in INT_FIELDS:
            bad |= present & value.mod(1).ne(0)
        if field in {"page", "last_page", "transaction_number_in_filing"}:
            bad |= present & value.lt(1)
        if field in SCORE_FIELDS:
            bad |= present & value.gt(100)
        if field == "source_year":
            bad |= present & value.ne(2025)
        return bad.fillna(False)
    if field in DATE_FIELDS:
        return present & pd.to_datetime(s.replace("", pd.NA), format="%Y-%m-%d", errors="coerce").isna()
    if field == "ticker_parse_status":
        return present & ~s.isin(["accepted", "not_applicable", "ambiguous_preserved"])
    return None

domain_results = []
for field in audit_input:
    bad = domain_failures(audit_input, field)
    if bad is not None:
        domain_results.append({"variable":field, "domain violations":int(bad.sum())})
display(pd.DataFrame(domain_results))
print("Only the displayed domains were machine-tested; source meanings were not inferred from these checks.")


,count
exact duplicates,0
duplicate filing+row keys,0
sentinel tokens,0
replacement character (encoding),0
leading-zero IDs,0
total/subtotal labels,0
nonempty amount min > max,0
review flagged,206


Non-standalone raw date strings: 640 of which scraper did not flag: 617


,filing_id,transaction_number_in_filing,transaction_date_raw,transaction_date,needs_review,original_pdf_url
56,20024346,27,01/13/2025 1/17/25 CLASS,2025-01-13,False,https://disclosures-clerk.house.gov/public_disc/ptr-pdfs/2025/20024346.pdf
181,20026517,3,12/04/2024 3.00% 07/01/2,2024-12-04,True,https://disclosures-clerk.house.gov/public_disc/ptr-pdfs/2025/20026517.pdf
183,20026533,1,12/16/2024 1,2024-12-16,False,https://disclosures-clerk.house.gov/public_disc/ptr-pdfs/2025/20026533.pdf
185,20026533,3,12/19/2024 1,2024-12-19,False,https://disclosures-clerk.house.gov/public_disc/ptr-pdfs/2025/20026533.pdf
187,20026533,5,12/27/2024 1,2024-12-27,False,https://disclosures-clerk.house.gov/public_disc/ptr-pdfs/2025/20026533.pdf
189,20026533,7,12/16/2024 1,2024-12-16,False,https://disclosures-clerk.house.gov/public_disc/ptr-pdfs/2025/20026533.pdf
191,20026533,9,12/16/2024 1,2024-12-16,False,https://disclosures-clerk.house.gov/public_disc/ptr-pdfs/2025/20026533.pdf
193,20026533,11,12/11/2024 1,2024-12-11,False,https://disclosures-clerk.house.gov/public_disc/ptr-pdfs/2025/20026533.pdf


Checked and not found under these definitions: exact duplicates; duplicate filing+row keys; sentinel tokens; replacement character (encoding); leading-zero IDs; total/subtotal labels; nonempty amount min > max
These token/pattern checks are limited screens, not proof that every form of encoding or missing-value defect is absent.
Transactions before 2025 retained: 504
Blank, exact, and ranged amounts are different states:


,rows
amount_status,
valid_range,7657
nonstandard_exact,8
missing_range_bound,2


Amount exceptions requiring source review:


,filing_id,transaction_number_in_filing,amount_raw,amount_exact,amount_status,original_pdf_url
1085,20027995,8,$800.00,800.0,nonstandard_exact,https://disclosures-clerk.house.gov/public_disc/ptr-pdfs/2025/20027995.pdf
2088,20030236,1,$823.45,823.45,nonstandard_exact,https://disclosures-clerk.house.gov/public_disc/ptr-pdfs/2025/20030236.pdf
3918,20030742,1,$647.63,647.63,nonstandard_exact,https://disclosures-clerk.house.gov/public_disc/ptr-pdfs/2025/20030742.pdf
5745,20032087,1,"$2,761.57",2761.57,nonstandard_exact,https://disclosures-clerk.house.gov/public_disc/ptr-pdfs/2025/20032087.pdf
5746,20032087,2,"$1,118.07",1118.07,nonstandard_exact,https://disclosures-clerk.house.gov/public_disc/ptr-pdfs/2025/20032087.pdf
6092,20032187,5,$459.77,459.77,nonstandard_exact,https://disclosures-clerk.house.gov/public_disc/ptr-pdfs/2025/20032187.pdf
6263,20032230,1,"$1,972.55",1972.55,nonstandard_exact,https://disclosures-clerk.house.gov/public_disc/ptr-pdfs/2025/20032230.pdf
6459,20033320,1,"$2,000.00",,missing_range_bound,https://disclosures-clerk.house.gov/public_disc/ptr-pdfs/2025/20033320.pdf
6503,20033370,2,$224.00,224.0,nonstandard_exact,https://disclosures-clerk.house.gov/public_disc/ptr-pdfs/2025/20033370.pdf
7331,20033581,1,"$1,140.00",,missing_range_bound,https://disclosures-clerk.house.gov/public_disc/ptr-pdfs/2025/20033581.pdf


Adjacent similar rows flagged (not automatically duplicates): 166


,variable,domain violations
0,source_year,0
1,transaction_number_in_filing,0
2,transaction_date,0
3,notification_date,0
4,amount_min,0
5,amount_max,0
6,amount_exact,0
7,page,0
8,last_page,0
9,page_recovery_used,0


Only the displayed domains were machine-tested; source meanings were not inferred from these checks.


**Check the exceptions:** Text beside a date may have bled in from an adjacent PDF cell. This does not automatically make the resolved ISO date wrong. Open the linked original PDF for examples before changing them. Explicitly record defect classes checked and not found above.

## 2. Data dictionary for the final P2 columns

The dictionary uses the final column names, including original-value backups and derived
checks. Its observed levels and missingness describe a preview of the exact transformations
logged next. The same function is used to build the final output, and equality is checked.
Expected domains come from field meaning and the pinned parser, not from treating observed
minimum/maximum values as universal limits. An empty extracted field does not establish
that the original PDF was empty.

**What to look for:** observed category levels come from this CSV, while the notes and expected ranges are interpretations to verify against the disclosure and parser code.

In [6]:
# Human outcomes extend the original parser review flags; they do not overwrite them.
# This single decision file also generates the website review and notebook cleaning log.
import json
review_decisions = json.loads(Path("data/review/decisions.json").read_text(encoding="utf-8"))
assert len({d["id"] for d in review_decisions}) == len(review_decisions)
for decision in review_decisions:
    assert decision["decision"] in {"keep", "correct"}
    for reviewed in decision["rows"]:
        mask = clean.filing_id.eq(decision["filing_id"]) & clean.transaction_number_in_filing.eq(reviewed["transaction_number_in_filing"])
        assert mask.sum() == 1
        original = clean.loc[mask].iloc[0]
        for field, value in {**reviewed["original_flags"], **reviewed["before"]}.items():
            assert original[field] == value, f"Review input changed: {decision['id']} / {field}"
        if decision["decision"] == "keep":
            assert reviewed["before"] == reviewed["after"]

# The transformation is deterministic and limited to Sean's approved record.
# It is previewed for dictionary completeness and used again in the cleaning section.
def build_p2(frame):
    result = frame.copy()
    s = result.transaction_date_raw
    result["raw_date_has_extra_text"] = s.ne("") & ~s.str.fullmatch(r"\d{1,2}/\d{1,2}/\d{4}")
    result["raw_date_prefix"] = s.str.extract(r"^(\d{1,2}/\d{1,2}/\d{4})", expand=False).fillna("")
    prefix = pd.to_datetime(result.raw_date_prefix.replace("", pd.NA), format="%m/%d/%Y", errors="coerce").dt.strftime("%Y-%m-%d").fillna("")
    result["date_prefix_disagrees"] = prefix.ne("") & prefix.ne(result.transaction_date)
    result["amount_exact_before_p2"] = result.amount_exact
    result["amount_status_before_p2"] = result.amount_status
    for decision in review_decisions:
        if decision["decision"] != "correct":
            continue
        for reviewed in decision["rows"]:
            selected = result.filing_id.eq(decision["filing_id"]) & result.transaction_number_in_filing.eq(reviewed["transaction_number_in_filing"])
            assert selected.sum() == 1
            for field, before in reviewed["before"].items():
                assert result.loc[selected, field].iloc[0] == before
                result.loc[selected, field] = reviewed["after"][field]
    return result

dictionary_data = build_p2(clean)
NOTES = {
"filing_id":"House filing identifier; not a person ID. Keep leading zeros.",
"politician":"Member name reported in the filing; not a stable person identifier.",
"member_status":"Reported filer role, not independently checked employment status.",
"state_district":"Reported state/district code; retain as text.",
"source_pdf":"Archived source filename, joined to data/raw/2025_pdfs/.",
"source_year":"Filing-index year 2025, not a transaction-date filter.",
"transaction_number_in_filing":"Extraction sequence; combine with filing_id for row key.",
"owner":"Source owner code; blank means no separately extracted code, not necessarily the member.",
"asset_v8_2_cleaned":"Resolver's asset display text; original kept in asset_v8_1 and PDF.",
"asset_v8_1":"Pre-resolver asset text. Restore asset_v8_2_cleaned from this to reverse resolver edits.",
"ticker_v8_2_cleaned":"Resolver ticker; a blank or symbol alone does not establish asset eligibility.",
"ticker_v8_1":"Ticker candidate from original extraction; retained for comparison.",
"asset_type":"Disclosure asset-type code; use the House code guide, not a guessed mapping.",
"transaction_type":"Reported P/S/E code, sometimes with a partial/full sale suffix.",
"transaction_date":"Reported trade date, not filing or notification date. Earlier years remain.",
"notification_date":"Reported notification date; not necessarily filing date.",
"amount_min":"Lower endpoint of a disclosed band; never substitute for the exact trade value.",
"amount_max":"Upper endpoint of a disclosed band; blank for exact amounts or unresolved ranges.",
"amount_exact":"Exact amount only when explicitly reported; one reviewed $2,000 value restored.",
"amount_category":"Source amount-band/text label; may contain a single exact amount.",
"amount_status":"Parser classification plus the one reviewed P2 reclassification.",
"filing_status":"Source status of the entry; not an independent duplicate decision.",
"subholding":"Reported account/subholding description; no separate entity resolution.",
"location":"Location text only when present/extracted; blank is not zero.",
"description":"Source transaction description; may distinguish superficially duplicate rows.",
"comments":"Source comments when present/extracted.",
"page":"First source PDF page for the extracted transaction (one-based).",
"last_page":"Last page contributing text; may differ for a continuation.",
"page_parse_method":"Parser route used; a method label is not an accuracy guarantee.",
"page_recovery_used":"Whether the parser used its page-recovery route.",
"filing_page_completeness_issue":"Parser suspects a page coverage problem; not proof of complete coverage when False.",
"suspected_missed_pages":"Parser's page identifiers needing review; blank means none flagged.",
"page_continuation_used":"Whether text continued across pages for this row.",
"duplicate_geometry_rows_merged":"Count of geometry fragments merged by upstream parsing, not proven duplicate trades removed.",
"possible_adjacent_same_signature":"Similar adjacent trade fields; retain unless source evidence justifies deletion.",
"row_shaded":"PDF shading detection; does not itself determine transaction meaning.",
"geometry_quality_score":"Parser heuristic score, not a probability of correctness.",
"unicode_repair_applied":"Parser normalized special font characters in extracted text.",
"unicode_repair_count":"Count reported by the parser's text normalization routine.",
"needs_review":"Original parser flag retained even after a P2 review; no flag is not proof of correctness.",
"original_pdf_url":"Official House PDF link; archival copy remains in data/raw.",
"review_level":"Parser review severity, not a final human verdict.",
"review_reason":"Original reasons kept as history; see the P2 log for reviewed outcomes.",
"owner_raw":"Extracted owner-cell text before field interpretation.",
"asset_raw":"Extracted asset-cell text; may include nearby fragments, not a PDF facsimile.",
"asset_lookup_context":"Source-derived text provided to ticker resolution.",
"transaction_type_raw":"Extracted type-cell text can include neighboring description fragments.",
"transaction_date_raw":"Extracted date-cell text; 2025-01-13 example includes option-description fragments.",
"notification_date_raw":"Extracted notification-cell text before date parsing.",
"amount_raw":"Extracted amount text retained unchanged beside parsed dollar fields.",
"continuation_raw":"Text attached by the continuation routine; blank when none attached.",
"detail_raw":"Whole extracted row/detail text; original layout remains in the PDF.",
"review_reason_v8_1":"Review reasons before ticker resolution.",
"review_level_v8_1":"Review severity before ticker resolution.",
"needs_review_v8_1":"Review flag before ticker resolution.",
"geometry_quality_score_v8_1":"Heuristic score before ticker resolution.",
"ticker_candidate_raw":"Resolver candidate when supplied; blank may mean none required.",
"ticker_parse_status":"accepted/not_applicable/ambiguous_preserved describe resolver decisions, not human verification.",
"ticker_validation_source":"Resolver's evidence-source label; blank may be not applicable.",
"ticker_changed":"Whether the resolver changed ticker text relative to V8.1.",
"asset_changed_by_ticker_resolver":"Whether the resolver changed asset text relative to V8.1.",
"raw_date_has_extra_text":"True for nonblank strings failing the date-only shape; False is not proof of a valid calendar date.",
"raw_date_prefix":"Leading MM/DD/YYYY-shaped token; separate calendar-validity parsing is required.",
"date_prefix_disagrees":"True only when a readable prefix differs from the normalized date; False can include unreadable prefixes.",
"amount_exact_before_p2":"Pre-P2 exact amount; use to reverse the manual correction.",
"amount_status_before_p2":"Pre-P2 amount classification; use to reverse the manual correction."
}
assert set(dictionary_data.columns) <= set(NOTES), "Every final column needs an explicit definition"
def expected_domain(x):
    if x in BOOL_FIELDS: return "True or False"
    if x == "source_year": return "2025 for this filing-index extract"
    if x in {"page", "last_page", "transaction_number_in_filing"}: return "positive integer"
    if x in INT_FIELDS: return "nonnegative integer"
    if x in MONEY_FIELDS: return "nonnegative USD or blank where not applicable/unresolved"
    if x in SCORE_FIELDS: return "0–100 heuristic points"
    if x in DATE_FIELDS: return "valid calendar date; no 2025-only restriction"
    if x == "raw_date_prefix": return "MM/DD/YYYY-shaped prefix or blank; calendar check separate"
    if x in {"amount_status", "amount_status_before_p2"}: return "valid_range, nonstandard_exact, missing_range_bound (observed parser classes)"
    if x == "ticker_parse_status": return "accepted, not_applicable, ambiguous_preserved"
    if x == "original_pdf_url": return "House public disclosure PDF URL"
    if x in CATEGORY_FIELDS: return "source/parser-defined codes; observed levels listed, not a universal codebook"
    return "source text or identifier; no numeric bounds"
def units(x):
    if x in MONEY_FIELDS: return "USD"
    if x in SCORE_FIELDS: return "heuristic points"
    if x in {"page", "last_page"}: return "PDF page number"
    if x == "source_year": return "filing-index calendar year"
    if x in INT_FIELDS: return "count or sequence index"
    if x in DATE_FIELDS: return "calendar date"
    return "not applicable (text/code/flag)"
def missing_meaning(x):
    if x in {"amount_min", "amount_max"}: return "No usable range endpoint: exact amount or unresolved range; inspect amount_status."
    if x in {"amount_exact", "amount_exact_before_p2"}: return "No extracted exact amount: usually a range, possibly unresolved."
    if x.startswith("ticker"): return "No extracted candidate/evidence, or not applicable; inspect asset_type and status."
    if x in {"review_reason", "review_level", "review_reason_v8_1", "review_level_v8_1", "suspected_missed_pages"}: return "No corresponding issue recorded by parser; not independent validation."
    if x == "raw_date_prefix": return "No leading date-shaped token found."
    return "Blank in extracted/derived field; consult original PDF before inferring absence."
dictionary_rows = []
for x in dictionary_data:
    s = dictionary_data[x].astype(str)
    observed = sorted(s.unique())
    levels = ", ".join(repr(v) for v in observed) if field_type(x) in {"category", "boolean"} else "Not categorical"
    bad = domain_failures(dictionary_data, x)
    contradiction = (f"{int(bad.sum())} violations of the implemented domain check; source meaning not independently validated."
                     if bad is not None else "No automatic semantic/codebook test for this field; see source notes and reviewed exceptions.")
    if x == "amount_exact_before_p2": contradiction = "Filing 20033320/1 was blank despite PDF showing $2,000; corrected in amount_exact."
    if x == "amount_status_before_p2": contradiction = "Filing 20033320/1 labeled missing_range_bound despite explicit exact amount."
    dictionary_rows.append({"variable":x, "type":field_type(x), "units":units(x), "factor levels":levels,
        "valid range":expected_domain(x), "missing count":int(s.eq("").sum()), "missing means":missing_meaning(x),
        "notes":NOTES[x], "documentation checks":contradiction})
dictionary = pd.DataFrame(dictionary_rows)
display(dictionary)


,variable,type,units,factor levels,valid range,missing count,missing means,notes,documentation checks
0,filing_id,string,not applicable (text/code/flag),Not categorical,source text or identifier; no numeric bounds,0,Blank in extracted/derived field; consult original PDF before inferring absence.,House filing identifier; not a person ID. Keep leading zeros.,No automatic semantic/codebook test for this field; see source notes and reviewed exceptions.
1,politician,string,not applicable (text/code/flag),Not categorical,source text or identifier; no numeric bounds,0,Blank in extracted/derived field; consult original PDF before inferring absence.,Member name reported in the filing; not a stable person identifier.,No automatic semantic/codebook test for this field; see source notes and reviewed exceptions.
2,member_status,category,not applicable (text/code/flag),"'Former Member', 'Member'","source/parser-defined codes; observed levels listed, not a universal codebook",0,Blank in extracted/derived field; consult original PDF before inferring absence.,"Reported filer role, not independently checked employment status.",No automatic semantic/codebook test for this field; see source notes and reviewed exceptions.
3,state_district,category,not applicable (text/code/flag),"'AL04', 'AL07', 'AR02', 'AR04', 'CA11', 'CA27', 'CA28', 'CA30', 'CA31', 'CA47', 'CA48', 'CA50', 'DC00', 'FL02', 'FL06', 'FL15', 'FL17', 'FL18', 'FL19', 'FL23', 'FL25', 'GA01', 'GA06', 'GA08', 'GA10', 'GA12', 'GA14', 'HI01', 'IL01', 'IL10', 'IN02', 'IN05', 'IN06', 'KY01', 'LA06', 'MA04', 'MA05', 'MA06', 'MA09', 'MD06', 'ME01', 'MI06', 'MI09', 'MI13', 'MN03', 'MN08', 'MO02', 'MS03', 'NC05', 'NC06', 'NC13', 'NC14', 'NJ01', 'NJ05', 'NJ07', 'NJ11', 'NV03', 'NY02', 'NY03', 'NY10', 'NY15', 'NY18', 'OH01', 'OH02', 'OH05', 'OH07', 'OH14', 'OK01', 'OR02', 'OR04', 'PA03', 'PA08', 'PA09', 'PA14', 'PA16', 'SC03', 'TN04', 'TN06', 'TN07', 'TN09', 'TX11', 'TX17', 'TX25', 'TX26', 'TX32', 'TX37', 'UT03', 'VA05', 'VA08', 'VA11', 'WA01', 'WA02', 'WA04', 'WA06', 'WA09', 'WV01'","source/parser-defined codes; observed levels listed, not a universal codebook",0,Blank in extracted/derived field; consult original PDF before inferring absence.,Reported state/district code; retain as text.,No automatic semantic/codebook test for this field; see source notes and reviewed exceptions.
4,source_pdf,string,not applicable (text/code/flag),Not categorical,source text or identifier; no numeric bounds,0,Blank in extracted/derived field; consult original PDF before inferring absence.,"Archived source filename, joined to data/raw/2025_pdfs/.",No automatic semantic/codebook test for this field; see source notes and reviewed exceptions.
5,source_year,integer,filing-index calendar year,Not categorical,2025 for this filing-index extract,0,Blank in extracted/derived field; consult original PDF before inferring absence.,"Filing-index year 2025, not a transaction-date filter.",0 violations of the implemented domain check; source meaning not independently validated.
6,transaction_number_in_filing,integer,count or sequence index,Not categorical,positive integer,0,Blank in extracted/derived field; consult original PDF before inferring absence.,Extraction sequence; combine with filing_id for row key.,0 violations of the implemented domain check; source meaning not independently validated.
7,owner,category,not applicable (text/code/flag),"'', 'DC', 'JT', 'SP'","source/parser-defined codes; observed levels listed, not a universal codebook",3799,Blank in extracted/derived field; consult original PDF before inferring absence.,"Source owner code; blank means no separately extracted code, not necessarily the member.",No automatic semantic/codebook test for this field; see source notes and reviewed exceptions.
8,asset_v8_2_cleaned,string,not applicable (text/code/flag),Not categorical,source text or identifier; no numeric bounds,0,Blank in extracted/derived field; consult original PDF before inferring absence.,Resolver's asset 

## 3. Quantified cleaning log and reviewed decisions

The log separates upstream PDF extraction, ticker resolution, and this milestone's decisions.
Stage 3 counts compare retained extracted text with parsed fields; they are not counts of
every low-level operation performed inside the parser. PDF layout and typography remain
recoverable in the preserved originals. Counts overlap and must not be summed as unique rows.

Alternatives considered: dropping every flagged row would discard usable disclosures;
deduplicating on asset/date/amount would remove two entries with different quantities;
assigning a standard dollar band to an exact amount would invent information. We retain
source-supported values and apply only the single explicitly reviewed correction.

This cell records transformations **with a count, reason, information loss, and reversal path**. Counts from different decisions can overlap; do not add them as a number of unique transactions.

**What to look for:** in the pinned batch the six direct Stage 4 comparisons are all zero. The date-prefix disagreement count is zero too, despite extra text in some raw date strings. The classification decisions still matter; inspect the original PDFs before making a manual correction.

In [7]:
# Before comparing two stages row by row, confirm they have the same number of rows.
assert len(raw)==len(clean),"Stage 4 unexpectedly changed row count"
# Confirm each row still has the same filing ID and position after Stage 4.
assert raw[keys].reset_index(drop=True).equals(clean[keys].reset_index(drop=True)),"Stage 4 changed transaction order or keys"
# Stop if the proposed key repeats; then one row might match more than one trade.
assert raw.duplicated(keys).sum()==0,"Resolve duplicate keys before accepting one-to-one comparison"
# Start an empty list to record what each stage did to the data.
changes=[]
# This helper adds one numbered entry to the log with a count, reason,
# what information was lost, and how to reverse the change.
def log_decision(stage,column,action,count,reason,lost,reverse):
    # Append a dictionary (one labeled record) to that list.
    changes.append({"#":len(changes)+1,"stage":stage,"column":column,"change made":action,"rows/cells affected":int(count),"why":reason,"what is lost":lost,"how to reverse":reverse})

# Stage 3 turns PDF text into useful fields. Compare raw asset, trade type,
# and date text with the parsed versions, while keeping both columns.
for original,parsed,reason in [
 ("asset_raw","asset","Separate the displayed asset from ticker and nearby PDF text for analysis"),
 ("transaction_type_raw","transaction_type","Isolate the P/S/E transaction code from adjacent PDF text"),
 ("transaction_date_raw","transaction_date","Convert source-like date text to an ISO date for chronological checks")]:
    # Count rows where the original and parsed text differ, then log that decision.
    count=int(raw[original].ne(raw[parsed]).sum())
    log_decision("3: PDF extraction",parsed,f"Parsed {original} into {parsed}",count,reason,f"Parsed field loses formatting/context; {original} retains extracted text and the PDF retains original layout.",f"Restore {parsed} from {original} to recover the retained extracted text; inspect original_pdf_url for original layout.")
# Count rows with a suggested ticker symbol extracted from the asset description.
log_decision("3: PDF extraction","ticker","Extracted ticker candidates from asset text",int(raw.ticker.ne("").sum()),"Source PDF asset text can contain a parenthetical stock symbol.","No raw text lost; asset_raw remains alongside ticker.","Drop the derived ticker to undo extraction; asset_raw and original_pdf_url retain its source evidence.")
# Count how many amounts fall into each parsing category, such as a normal range
# or an amount the parser could not confidently turn into two bounds.
for status,count in raw.amount_status.value_counts(dropna=False).items():
    reasons={"valid_range":"Two source dollar values match a recognized disclosure band.","nonstandard_exact":"The source displays a nonstandard exact amount rather than a standard band.","missing_range_bound":"The source range cannot safely supply both numeric bounds."}
    log_decision("3: PDF extraction","amount_status",f"Classified amount as {status}",count,reasons.get(status,"Amount parser classified source text."),"Raw amount text remains in amount_raw; numeric bounds may be absent when uncertain.","Drop amount classification/bound fields to undo derivation; amount_raw and original_pdf_url preserve source evidence.")

# Stage 4 records whether a ticker candidate was accepted, did not apply,
# or was too uncertain. A classification may leave the actual ticker text unchanged.
for status,count in clean.ticker_parse_status.value_counts(dropna=False).items():
    reasons={"accepted":"A source-structured ticker candidate was accepted.","not_applicable":"The row is not a stock asset, so stock ticker resolution does not apply.","ambiguous_preserved":"The candidate is too uncertain to assert as a ticker."}
    log_decision("4: ticker resolver","ticker_parse_status",f"Classified ticker as {status}",count,reasons.get(status,"Resolver classification; inspect pinned Stage 4 code."),"No source evidence lost; V8.1 ticker and asset are retained.","Drop the resolver classification fields; restore ticker/asset values from ticker_v8_1 and asset_v8_1 if needed.")
# Compare six Stage 4 fields to copies of their Stage 3 values.
# .ne() means "not equal"; summing True values counts differences.
actual_changes={current:int(clean[current].ne(clean[prior]).sum()) for current,prior in [
 ("ticker_v8_2_cleaned","ticker_v8_1"),("asset_v8_2_cleaned","asset_v8_1"),
 ("review_reason","review_reason_v8_1"),("review_level","review_level_v8_1"),
 ("needs_review","needs_review_v8_1"),("geometry_quality_score","geometry_quality_score_v8_1")]}
# Print the six counts. Zero is a useful result: the value stayed the same.
print("Stage 4 values changed (zero is a legitimate finding):",actual_changes)
# If any field really changed, record it and point to its saved older version.
for field,count in actual_changes.items():
    if count:
        prior={"ticker_v8_2_cleaned":"ticker_v8_1","asset_v8_2_cleaned":"asset_v8_1","review_reason":"review_reason_v8_1","review_level":"review_level_v8_1","needs_review":"needs_review_v8_1","geometry_quality_score":"geometry_quality_score_v8_1"}[field]
        log_decision("4: ticker resolver",field,"Replaced V8.1 value",count,"Resolver accepted a candidate or recalculated a ticker-specific review flag.",f"Nothing; {prior} retains the previous value.",f"Restore directly from {prior}.")
# Six original columns are kept beside the new versions. The count here is
# cells (rows times six), not the number of distinct trades.
log_decision("4: ticker resolver","six V8.1 audit columns","Preserved original fields next to resolved values",len(clean)*6,"Readers need a reversible before/after record.","Nothing; columns are added copies.","Drop the six *_v8_1 columns to reverse this addition.")

# Make a separate copy for our Part 2 checks and the approved manual correction.
p2 = build_p2(clean)
assert p2.equals(dictionary_data)
source_date = p2.transaction_date_raw
# Log the three Part 2 additions. The original text and cleaned date are still kept,
# so each flag can be checked or removed later.
log_decision("P2: audit","raw_date_has_extra_text","Flagged non-standalone source date text",int(p2.raw_date_has_extra_text.sum()),"Adjacent PDF text may be attached to a date and needs source review.","Nothing; original transaction_date_raw remains unchanged.","Remove the flag; compare original text with the source PDF.")
log_decision("P2: audit","raw_date_prefix","Extracted leading date token for checking",int(p2.raw_date_prefix.ne("").sum()),"Check whether the V8.1 date agrees with the visible leading date despite extra text.","Nothing; source-like text remains in transaction_date_raw.","Drop this derived column; extract again from transaction_date_raw.")
log_decision("P2: audit","date_prefix_disagrees","Flagged parsed date disagreements",int(p2.date_prefix_disagrees.sum()),"Any disagreement between source-like date prefix and normalized date warrants source review.","Nothing; both original and normalized dates remain.","Remove the derived flag and recompute from the two date fields.")
# Record preservation, retention judgments, and the one actual correction.
log_decision("P2: preservation", "amount_exact_before_p2, amount_status_before_p2", "Saved pre-P2 values", len(p2)*2,
    "An individual correction must be reversible without rerunning the parser.", "Nothing; two copies added.",
    "Restore amount_exact and amount_status from these columns; then drop the two backup columns if desired.")
# Generate the required decision log directly from the same original-flag reviews.
for decision in review_decisions:
    changed_cells = sum(row["before"][field] != value for row in decision["rows"] for field, value in row["after"].items())
    count = changed_cells if decision["decision"] == "correct" else len(decision["rows"])
    assert count == decision["affected_count"]
    log_decision("P2: manual correction" if decision["decision"] == "correct" else "P2: manual retention",
                 decision["column"], decision["action"], count, decision["reason"],
                 decision["what_is_lost"], decision["how_to_reverse"])
corrected = p2.filing_id.eq("20033320") & p2.transaction_number_in_filing.eq("1")
metadata_added = sorted(set(clean.columns) - set(raw.columns) - set(x for x in clean if x.endswith("_v8_1")) - {"asset_v8_2_cleaned", "ticker_v8_2_cleaned"})
log_decision("4: resolver schema", ", ".join(metadata_added), "Added resolver decision metadata; renamed asset/ticker to *_v8_2_cleaned", len(clean)*len(metadata_added),
    "Keep decisions inspectable and distinguish resolved display values from preserved inputs.",
    "Nothing; asset_v8_1 and ticker_v8_1 retain pre-resolver text.",
    "Drop these resolver-only metadata columns, rename asset_v8_2_cleaned to asset and ticker_v8_2_cleaned to ticker; use *_v8_1 values to undo any value changes.")

# Choose a new file under data/clean for the finished Part 2 table.
output=Path("data/clean/house_ptr_2025_p2.csv")
# Create that folder if it is missing.
output.parent.mkdir(parents=True,exist_ok=True)
# Save the table as CSV without adding an extra row-number column.
p2.to_csv(output,index=False)
# Turn the list of logged decisions into a table and display it.
log_df=pd.DataFrame(changes)
display(log_df)
# Show where the file was saved and the two date-check counts.
print("P2 clean output:",output)
print("Raw date extra-text flags:",int(p2.raw_date_has_extra_text.sum()),"date-prefix disagreements:",int(p2.date_prefix_disagrees.sum()))
output.parent.mkdir(parents=True, exist_ok=True)
log_df.to_csv(output.parent / "cleaning_log.csv", index=False)
dictionary.to_csv(output.parent / "data_dictionary.csv", index=False)
audit.to_csv(output.parent / "audit_inventory.csv", index=False)
print("Logged decisions:", len(log_df))
print("Manual correction: 1 transaction row, 2 value cells. Original flags remain as history.")
display(p2.loc[corrected, ["filing_id", "transaction_number_in_filing", "amount_raw", "amount_exact_before_p2", "amount_exact", "amount_status_before_p2", "amount_status", "needs_review", "review_reason"]])

# Summary of human outcomes alongside existing parser flags (no new flag system).
reviewed_keys = {(d["filing_id"], r["transaction_number_in_filing"]): d["decision"]
                 for d in review_decisions for r in d["rows"]}
flagged_keys = set(map(tuple, clean.loc[clean.needs_review.eq("True"), keys].to_numpy()))
print("Original parser-flagged rows:", len(flagged_keys))
print("Flagged rows with a recorded human decision:", len(flagged_keys & reviewed_keys.keys()))
print("Flagged rows not yet reviewed:", len(flagged_keys - reviewed_keys.keys()))
print("Additional reviewed rows without an original parser flag:", len(reviewed_keys.keys() - flagged_keys))
display(pd.DataFrame([{"filing_id":d["filing_id"], "transaction_number_in_filing":r["transaction_number_in_filing"],
                      **r["original_flags"], "human_decision":d["decision"], "reason":d["reason"]}
                     for d in review_decisions for r in d["rows"]]))


Stage 4 values changed (zero is a legitimate finding): {'ticker_v8_2_cleaned': 0, 'asset_v8_2_cleaned': 0, 'review_reason': 0, 'review_level': 0, 'needs_review': 0, 'geometry_quality_score': 0}


,#,stage,column,change made,rows/cells affected,why,what is lost,how to reverse
0,1,3: PDF extraction,asset,Parsed asset_raw into asset,7601,Separate the displayed asset from ticker and nearby PDF text for analysis,Parsed field loses formatting/context; asset_raw retains extracted text and the PDF retains original layout.,Restore asset from asset_raw to recover the retained extracted text; inspect original_pdf_url for original layout.
1,2,3: PDF extraction,transaction_type,Parsed transaction_type_raw into transaction_type,2731,Isolate the P/S/E transaction code from adjacent PDF text,Parsed field loses formatting/context; transaction_type_raw retains extracted text and the PDF retains original layout.,Restore transaction_type from transaction_type_raw to recover the retained extracted text; inspect original_pdf_url for original layout.
2,3,3: PDF extraction,transaction_date,Parsed transaction_date_raw into transaction_date,7667,Convert source-like date text to an ISO date for chronological checks,Parsed field loses formatting/context; transaction_date_raw retains extracted text and the PDF retains original layout.,Restore transaction_date from transaction_date_raw to recover the retained extracted text; inspect original_pdf_url for original layout.
3,4,3: PDF extraction,ticker,Extracted ticker candidates from asset text,6923,Source PDF asset text can contain a parenthetical stock symbol.,No raw text lost; asset_raw remains alongside ticker.,Drop the derived ticker to undo extraction; asset_raw and original_pdf_url retain its source evidence.
4,5,3: PDF extraction,amount_status,Classified amount as valid_range,7657,Two source dollar values match a recognized disclosure band.,Raw amount text remains in amount_raw; numeric bounds may be absent when uncertain.,Drop amount classification/bound fields to undo derivation; amount_raw and original_pdf_url preserve source evidence.
5,6,3: PDF extraction,amount_status,Classified amount as nonstandard_exact,8,The source displays a nonstandard exact amount rather than a standard band.,Raw amount text remains in amount_raw; numeric bounds may be absent when uncertain.,Drop amount classification/bound fields to undo derivation; amount_raw and original_pdf_url preserve source evidence.
6,7,3: PDF extraction,amount_status,Classified amount as missing_range_bound,2,The source range cannot safely supply both numeric bounds.,Raw amount text remains in amount_raw; numeric bounds may be absent when uncertain.,Drop amount classification/bound fields to undo derivation; amount_raw and original_pdf_url preserve source evidence.
7,8,4: ticker resolver,ticker_parse_status,Classified ticker as accepted,6830,A source-structured ticker candidate was accepted.,No source evidence lost; V8.1 ticker and asset are retained.,Drop the resolver classification fields; restore ticker/asset values from ticker_v8_1 and asset_v8_1 if needed.
8,9,4: ticker resolver,ticker_parse_status,Classified ticker as not_applicable,837,"The row is not a stock asset, so stock ticker resolution does not apply.",No source evidence lost; V8.1 ticker and asset are retained.,Drop the resolver classification fields; restore ticker/asset values from ticker_v8_1 and asset_v8_1 if needed.
9,10,4: ticker resolver,six V8.1 audit columns,Preserved original fields next to resolved values,46002,Readers need a reversible before/after record.,Nothing; columns are added copies.,Drop the six *_v8_1 columns to reverse this addition.


P2 clean output: data\clean\house_ptr_2025_p2.csv
Raw date extra-text flags: 640 date-prefix disagreements: 0
Logged decisions: 19
Manual correction: 1 transaction row, 2 value cells. Original flags remain as history.


,filing_id,transaction_number_in_filing,amount_raw,amount_exact_before_p2,amount_exact,amount_status_before_p2,amount_status,needs_review,review_reason
6459,20033320,1,"$2,000.00",,2000.0,missing_range_bound,nonstandard_exact,True,missing_range_bound


Original parser-flagged rows: 206
Flagged rows with a recorded human decision: 3
Flagged rows not yet reviewed: 203
Additional reviewed rows without an original parser flag: 2


,filing_id,transaction_number_in_filing,needs_review,review_level,review_reason,possible_adjacent_same_signature,human_decision,reason
0,20024346,27,False,,,False,keep,PDF page 4 places 01/13/2025 in the transaction-date cell; 01/17/25 belongs to the option description.
1,20026517,3,True,low,adjacent transaction has same core signature,True,keep,"PDF page 1 lists 15,000 units and 5,000 units separately; matching asset/date/band alone is insufficient for deletion."
2,20026517,4,False,,,False,keep,"PDF page 1 lists 15,000 units and 5,000 units separately; matching asset/date/band alone is insufficient for deletion."
3,20027995,8,True,low,nonstandard_exact,False,keep,PDF page 2 explicitly reports $800.00; inventing a disclosure range would lose precision.
4,20033320,1,True,high,missing_range_bound,False,correct,"PDF page 1 explicitly reports $2,000.00. Leaving it unknown discards readable evidence; inferring a range invents a bound."


The P2 output adds three date-check fields and two pre-P2 amount backups. Nonblank date text failing the standalone-date shape is flagged; an invalid date can still have the right shape, so calendar parsing is a separate audit. The original review flags remain historical evidence even for manually reviewed rows.

The $2,000 correction changes exactly two cells in one transaction. The source PDFs and Stage 3/4 CSVs are unchanged. The three retention decisions concern four other transaction rows; retaining those rows does not clear all similar flags across the dataset. The second missing-range case remains unresolved and flagged. No near-duplicate rows are deleted.

## 4. Row and column accounting

The tables below reconcile rows and column names, then verify the single-row correction and preservation of the source PDFs. Counts are computed, not hand-entered.

In [8]:
# Compare column names in the first and final CSVs.
# A name that disappears may have been renamed, with its values kept elsewhere.
removed=sorted(set(raw.columns)-set(p2.columns))
# List names that appear in the final table but were absent from Stage 3.
added=sorted(set(p2.columns)-set(raw.columns))
# Build a before-and-after count of rows and columns. This notebook removes zero rows.
accounting=pd.DataFrame([("Raw V8.1 rows",len(raw)),("Exact duplicate rows removed",0),("Near-duplicate rows removed",0),("Other rows removed",0),("P2 clean rows",len(p2)),("Raw columns",raw.shape[1]),("Columns removed/renamed",len(removed)),("Columns added/renamed",len(added)),("P2 clean columns",p2.shape[1])],columns=["item","count"])
# Stop if the final row count changed unexpectedly.
assert len(raw)==len(p2)
# Check that old columns minus absent names plus new names equals the final count.
assert raw.shape[1]-len(removed)+len(added)==p2.shape[1]
# Show the counts, the column names, and the paths to each CSV for the assignment.
display(accounting)
print("Removed or renamed:",removed)
print("Added or renamed:",added)
print("Raw V8.1 CSV:",source)
print("V8.2 CSV:",resolved)
print("P2 clean CSV:",output)
renames = {"asset": "asset_v8_2_cleaned", "ticker": "ticker_v8_2_cleaned"}
genuinely_dropped = [x for x in removed if x not in renames]
assert not genuinely_dropped
print("Explicit renames:", renames)
print("Columns truly dropped:", genuinely_dropped)
print("Pure column additions:", sorted(set(added) - set(renames.values())))
assert p2.shape[1] == clean.shape[1] + 5
assert set(p2.columns) == set(dictionary.variable)
assert (p2.amount_exact != clean.amount_exact).sum() == 1
assert (p2.amount_status != clean.amount_status).sum() == 1
unchanged = [x for x in clean if x not in {"amount_exact", "amount_status"}]
assert p2[unchanged].equals(clean[unchanged])
assert p2.amount_exact_before_p2.equals(clean.amount_exact)
assert p2.amount_status_before_p2.equals(clean.amount_status)
assert p2.duplicated(keys).sum() == 0
roundtrip = pd.read_csv(output, dtype=str, keep_default_na=False)
assert roundtrip.shape == p2.shape
assert roundtrip.loc[corrected, "amount_exact"].iloc[0] == "2000.0"
raw_hashes_after = {str(p): file_hash(Path(p)) for p in raw_hashes_before}
assert raw_hashes_before == raw_hashes_after
assert set(raw_hashes_before) == {str(p) for p in Path("data/raw").rglob("*") if p.is_file()}
print("Raw-file integrity: all", len(raw_hashes_before), "files unchanged across this notebook run.")
print("Tidy output: one extracted transaction per row; each field is a column; original PDF preserved.")
print("Coverage: archived PDFs", len(list(Path("data/raw/2025_pdfs").glob("*.pdf"))),
      "filings with extracted rows", raw.filing_id.nunique(), "transaction rows", len(p2))
unresolved_amounts = p2.loc[p2.amount_status.eq("missing_range_bound"), keys + ["amount_raw", "original_pdf_url"]]
print("Unresolved missing-range cases retained:", len(unresolved_amounts))
display(unresolved_amounts)
accounting.to_csv(output.parent / "row_column_accounting.csv", index=False)
import json, platform, importlib.metadata
validation = {"rows":len(p2), "raw_columns":raw.shape[1], "resolved_columns":clean.shape[1], "p2_columns":p2.shape[1],
 "dictionary_fields":len(dictionary), "log_decisions":len(log_df), "corrected_rows":1, "corrected_value_cells":2,
 "reviewed_rows":5, "raw_files_unchanged":len(raw_hashes_before), "raw_date_flags":int(p2.raw_date_has_extra_text.sum()),
 "date_disagreements":int(p2.date_prefix_disagrees.sum()), "unresolved_amount_rows":len(unresolved_amounts),
 "not_found_checks":not_found, "python":platform.python_version(),
 "packages":{pkg:importlib.metadata.version(pkg) for pkg in ["pandas", "numpy", "PyMuPDF"]}}
Path("data/clean/validation_summary.json").write_text(json.dumps(validation, indent=2), encoding="utf-8")
display(pd.Series(validation, name="verified result").to_frame())


,item,count
0,Raw V8.1 rows,7667
1,Exact duplicate rows removed,0
2,Near-duplicate rows removed,0
3,Other rows removed,0
4,P2 clean rows,7667
5,Raw columns,50
6,Columns removed/renamed,2
7,Columns added/renamed,18
8,P2 clean columns,66


Removed or renamed: ['asset', 'ticker']
Added or renamed: ['amount_exact_before_p2', 'amount_status_before_p2', 'asset_changed_by_ticker_resolver', 'asset_v8_1', 'asset_v8_2_cleaned', 'date_prefix_disagrees', 'geometry_quality_score_v8_1', 'needs_review_v8_1', 'raw_date_has_extra_text', 'raw_date_prefix', 'review_level_v8_1', 'review_reason_v8_1', 'ticker_candidate_raw', 'ticker_changed', 'ticker_parse_status', 'ticker_v8_1', 'ticker_v8_2_cleaned', 'ticker_validation_source']
Raw V8.1 CSV: data\work\04_transactions\transactions_raw.csv
V8.2 CSV: data\work\04_transactions\transactions_resolved.csv
P2 clean CSV: data\clean\house_ptr_2025_p2.csv
Explicit renames: {'asset': 'asset_v8_2_cleaned', 'ticker': 'ticker_v8_2_cleaned'}
Columns truly dropped: []
Pure column additions: ['amount_exact_before_p2', 'amount_status_before_p2', 'asset_changed_by_ticker_resolver', 'asset_v8_1', 'date_prefix_disagrees', 'geometry_quality_score_v8_1', 'needs_review_v8_1', 'raw_date_has_extra_text', 'raw_date

Raw-file integrity: all 518 files unchanged across this notebook run.
Tidy output: one extracted transaction per row; each field is a column; original PDF preserved.
Coverage: archived PDFs 515 filings with extracted rows 449 transaction rows 7667
Unresolved missing-range cases retained: 1


,filing_id,transaction_number_in_filing,amount_raw,original_pdf_url
7331,20033581,1,"$1,140.00",https://disclosures-clerk.house.gov/public_disc/ptr-pdfs/2025/20033581.pdf


,verified result
rows,7667
raw_columns,50
resolved_columns,61
p2_columns,66
dictionary_fields,66
log_decisions,19
corrected_rows,1
corrected_value_cells,2
reviewed_rows,5
raw_files_unchanged,518


## 5. Provenance brief (166 words)

This dataset contains 7,667 reported transactions extracted from 449 of 515 financial-disclosure PDFs archived from the U.S. House Clerk’s 2025 filing index. House members file these reports to disclose financial transactions; the Clerk publishes them. Each row represents one extracted transaction linked to its source PDF.

The pipeline reads digital PDFs with selectable text. Handwritten or scanned forms need a more complex approach, such as optical character recognition (OCR), which this pipeline does not perform. The 66 PDFs that produced no transaction rows remain archived but contribute no rows to this dataset; this does not mean they report no transactions.

The selection follows filing-index year, so some transactions occurred before 2025. Most amounts are dollar ranges; some are exact amounts. Original PDFs are preserved, and corrections are documented.

The dataset cannot establish unreported trades, exact values where only ranges were disclosed, or who personally made investment decisions. Extraction can miss or misread information. Selected examples were manually reviewed; the entire dataset has not been individually verified.


## Submission and review record

- [x] Scope and four selected source-review cases confirmed by Sean in chat (five rows).
- [x] Approved provenance wording included.
- [x] Correction, original-value backups, and reversal instructions implemented.
- [x] Sean confirmed all five self-scores as 4 — Excellent on September 26, 2026. See [self-scored rubric](https://github.com/StrokeOfLuck/dsa405-part-2/blob/main/SELF_ASSESSMENT.md) for evidence and limitations; this is a self-assessment, not an awarded grade.
- [x] Bench Check 1 completed September 26, 2026, as reported by Sean.
- [x] Overall exceptions and limitations review confirmed by Sean on September 26, 2026. Unresolved flags are retained; 203 rows remain individually unreviewed.
- [ ] Final check of the saved notebook before upload.
- [ ] Submit the notebook, repository link, and self-scored rubric to Moodle.

**AI use:** ChatGPT (Codex) assisted with notebook code, audit documentation, execution,
and interpretation. Sean personally reviewed the four shown source-PDF cases and approved
the decisions and provenance wording. That approval does not claim individual review of all
transactions. The parser left a readable $2,000 amount blank; checking its original PDF exposed
the error, and the notebook preserves both the old and corrected values. Sean remains responsible
for understanding the code and reviewing the final self-assessment.

**Sources:** [assignment](https://github.com/jon-holt/DSA-405-Student/blob/main/assignments/projects/DSA405_P2_AuditCleaningLog_FA26.md),
[rubric](https://github.com/jon-holt/DSA-405-Student/blob/main/course/DSA405_ProjectRubrics_FA26.md),
[review record](https://github.com/StrokeOfLuck/dsa405-part-2/blob/main/REVIEW_PROGRESS.md).


**Execution evidence:** The saved code outputs below the required sections are produced by
a fresh kernel. PDF hashes are checked before and after execution. A first run in a fresh
working folder rebuilt the extraction from all archived PDFs; a later validation rerun can
reuse the parser checkpoint while rerunning the notebook's audit and correction logic.
